In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, desc, lit, when, round as spark_round
import time

spark = SparkSession.builder \
    .appName("M16-Lab03-Transformations-and-Actions") \
    .master("local[*]") \
    .getOrCreate()

print(f"✅ Spark {spark.version} running in local mode")


In [ ]:
import random

random.seed(42)

data = []
users = [f"U{i:04d}" for i in range(1, 501)]
songs = [f"S{i:03d}" for i in range(1, 51)]
artists = [f"A{i:02d}" for i in range(1, 11)]
statuses = ["completed", "completed", "completed", "skipped", "error"]

for i in range(100_000):
    data.append((
        f"P{i:06d}",
        random.choice(users),
        random.choice(songs),
        random.choice(artists),
        f"2025-03-{random.randint(1, 31):02d}",
        random.choice(statuses),
        random.randint(30, 300)
    ))

columns = ["play_id", "user_id", "song_id", "artist_id", "play_date", "status", "duration_seconds"]
df = spark.createDataFrame(data, columns)

print(f"✅ Test dataset created: {df.count()} rows, {len(df.columns)} columns")
df.printSchema()


In [ ]:
print("=" * 60)
print("EXPERIMENT 1: Proving Transformations Are Lazy")
print("=" * 60)

start = time.time()

step1 = df.filter(col("status") == "completed")
step2 = step1.filter(col("duration_seconds") > 120)
step3 = step2.groupBy("artist_id").agg(
    count("play_id").alias("play_count"),
    spark_sum("duration_seconds").alias("total_seconds")
)
step4 = step3.withColumn("avg_seconds", spark_round(col("total_seconds") / col("play_count"), 1))
step5 = step4.orderBy(desc("play_count"))

elapsed = time.time() - start

print(f"\n5 transformations chained in {elapsed:.4f} seconds")
print(f"Type of step5: {type(step5)}")
print("Did Spark read the data? NO — these are just plans!")


In [ ]:
start = time.time()

step5.show(5)

elapsed = time.time() - start

print(f"\n.show() triggered execution in {elapsed:.4f} seconds")
print("NOW Spark read the data, filtered, grouped, aggregated, and sorted!")


In [ ]:
print("\n📋 OBSERVATIONS:")
print("1. Transformation chain time: _____ seconds (near zero)")
print("2. Action (.show) time:       _____ seconds (much longer)")
print("3. This proves transformations are lazy — they build a plan, not results")


In [ ]:
print("=" * 60)
print("EXPERIMENT 2: Inspecting Execution Plans")
print("=" * 60)

simple = df.filter(col("status") == "completed").select("play_id", "song_id", "duration_seconds")

print("\n--- Simple Plan (filter + select) ---")
simple.explain()


In [ ]:
complex_query = (
    df.filter(col("status") == "completed")
      .filter(col("duration_seconds") > 60)
      .groupBy("artist_id")
      .agg(
          count("play_id").alias("play_count"),
          spark_sum("duration_seconds").alias("total_seconds")
      )
      .filter(col("play_count") > 100)
      .orderBy(desc("total_seconds"))
)

print("\n--- Complex Plan (filter + group + filter + sort) ---")
complex_query.explain()


In [ ]:
print("\n--- Extended Plan (all optimization stages) ---")
complex_query.explain(True)


In [ ]:
start = time.time()
complex_query.explain()
explain_time = time.time() - start

start = time.time()
complex_query.show(5)
show_time = time.time() - start

print(f"\n.explain() time: {explain_time:.4f} seconds (plan only, no execution)")
print(f".show() time:    {show_time:.4f} seconds (full execution)")
print(f"\n✅ .explain() is safe — it shows the plan without running it")


In [ ]:
What to look for in the plan:
- "FileScan" or "Scan" → reading data
- "Filter" → filtering rows
- "Project" → selecting columns (column pruning)
- "HashAggregate" → groupBy + aggregation
- "Sort" → orderBy
- "Exchange" → shuffle (data movement between nodes)


In [ ]:
print("=" * 60)
print("EXPERIMENT 3: Multiple Actions = Multiple Executions")
print("=" * 60)

pipeline = (
    df.filter(col("status") == "completed")
      .groupBy("song_id")
      .agg(count("play_id").alias("play_count"))
      .orderBy(desc("play_count"))
)


In [ ]:
print("\nAction 1: .count()")
start = time.time()
total = pipeline.count()
t1 = time.time() - start
print(f"  Result: {total} songs | Time: {t1:.4f}s")

print("\nAction 2: .show()")
start = time.time()
pipeline.show(5)
t2 = time.time() - start
print(f"  Time: {t2:.4f}s")

print("\nAction 3: .collect()")
start = time.time()
result = pipeline.collect()
t3 = time.time() - start
print(f"  Result: {len(result)} rows | Time: {t3:.4f}s")

print(f"\n📋 TOTAL TIME: {t1 + t2 + t3:.4f}s")
print(f"Each action re-executed the FULL pipeline (read → filter → group → sort)")
print(f"The pipeline ran 3 TIMES!")


In [ ]:
print("=" * 60)
print("EXPERIMENT 4: Caching to Avoid Re-Execution")
print("=" * 60)

pipeline_cached = (
    df.filter(col("status") == "completed")
      .groupBy("song_id")
      .agg(count("play_id").alias("play_count"))
      .orderBy(desc("play_count"))
)

pipeline_cached.cache()
print("✅ .cache() called — results will be stored after first action")

print("\nAction 1: .count() (first action — full execution + cache)")
start = time.time()
total = pipeline_cached.count()
t1_cached = time.time() - start
print(f"  Result: {total} songs | Time: {t1_cached:.4f}s")

print("\nAction 2: .show() (reads from cache)")
start = time.time()
pipeline_cached.show(5)
t2_cached = time.time() - start
print(f"  Time: {t2_cached:.4f}s")

print("\nAction 3: .collect() (reads from cache)")
start = time.time()
result = pipeline_cached.collect()
t3_cached = time.time() - start
print(f"  Result: {len(result)} rows | Time: {t3_cached:.4f}s")

print(f"\n📋 TOTAL TIME WITH CACHE: {t1_cached + t2_cached + t3_cached:.4f}s")


In [ ]:
total_no_cache = t1 + t2 + t3
total_with_cache = t1_cached + t2_cached + t3_cached

print("\n" + "=" * 60)
print("CACHE COMPARISON")
print("=" * 60)
print(f"Without cache: {total_no_cache:.4f}s (3 full executions)")
print(f"With cache:    {total_with_cache:.4f}s (1 full + 2 cache reads)")
print(f"Speedup:       {total_no_cache / total_with_cache:.1f}x faster")

print("\n📋 KEY INSIGHT:")
print("Cache when a DataFrame is reused across multiple actions.")
print("Don't cache DataFrames used only once.")


In [ ]:
pipeline_cached.unpersist()
print("✅ Cache cleared")


In [ ]:
print("=" * 60)
print("EXPERIMENT 5: Classify 15 Operations")
print("=" * 60)

classifications = [
    ("df.filter(col('x') > 10)", "?"),
    ("df.select('a', 'b')", "?"),
    ("df.groupBy('x').count()", "?"),
    ("df.show()", "?"),
    ("df.count()", "?"),
    ("df.collect()", "?"),
    ("df.join(df2, 'key')", "?"),
    ("df.orderBy('x')", "?"),
    ("df.write.parquet('path')", "?"),
    ("df.withColumn('y', col('x') * 2)", "?"),
    ("df.distinct()", "?"),
    ("df.take(5)", "?"),
    ("df.explain()", "?"),
    ("df.union(df2)", "?"),
    ("df.first()", "?"),
]

print(f"\n{'Operation':<45} {'Your Answer':<15} {'Correct':<15}")
print("-" * 75)

answers = ["T", "T", "T", "A", "A", "A", "T", "T", "A", "T", "T", "A", "Neither", "T", "A"]

for (op, _), answer in zip(classifications, answers):
    print(f"{op:<45} {'___':<15} {answer:<15}")


In [ ]:
print("\n📋 CLASSIFICATION RULES:")
print("1. Returns a DataFrame → Transformation (lazy)")
print("2. Returns data to driver or writes to storage → Action (eager)")
print("3. .explain() is special — shows plan, does NOT execute")
print("4. .cache() is special — marks for caching, no execution")
